In [26]:
import time 
import json
from openai import OpenAI
from dotenv import load_dotenv
import os

In [27]:
# =====================================================
# OPENAI CLIENT
# =====================================================
# Single client instance used by all agents.
#
# Future:
# This can be moved into:
#
# backend/services/llm_service.py
#
# =====================================================

client = OpenAI(
    api_key=os.environ.get("OPENAI_API_KEY"),
     base_url="https://aibe.mygreatlearning.com/openai/v1"
)

In [28]:
# =====================================================
# TECHNOLOGY DETECTOR
# =====================================================
#
# Purpose:
# Identify which technology stack the user
# is asking code for.
#
# Example:
#
# Input:
# "Generate Spring Boot CRUD API"
#
# Output:
# "spring"
#
# This helps us select the correct prompt
# template before calling the LLM.
#
# =====================================================

def detect_technology(
    user_prompt
):

    prompt = user_prompt.lower()

    technologies = {

        "spring": "spring",

        "spring boot": "spring",

        "fastapi": "fastapi",

        "django": "django",

        "react": "react",

        "docker": "docker",

        "sql": "sql",

        "test": "testing",

        "unit test": "testing"
    }

    for keyword, technology in technologies.items():

        if keyword in prompt:

            return technology

    return "generic"

In [29]:
# =====================================================
# PROMPT TEMPLATES
# =====================================================
#
# Purpose:
# Create specialized prompts for each technology.
#
# Why?
#
# Generic prompts produce inconsistent output.
#
# Technology-specific prompts produce:
#
# - Better structure
# - Better architecture
# - Better coding practices
#
# Future:
#
# Add templates for:
#
# - Kubernetes
# - Kafka
# - Terraform
# - Jenkins
# - Pytest
# - JUnit
#
# =====================================================

PROMPT_TEMPLATES = {

    "spring" : """
You are a Principal Spring Boot Architect.

Generate a complete production-ready Spring Boot application.

Include:

1. pom.xml
2. Entity
3. DTO
4. Repository
5. Service Interface
6. Service Implementation
7. REST Controller
8. Global Exception Handler
9. Validation
10. application.yml
11. Swagger Configuration
12. JUnit Test Cases

Requirements:

{requirement}

Return only code.

Use Java 21.

Use Spring Boot 3.

Use Constructor Injection.

Use Lombok.

Use DTO Pattern.

Use ResponseEntity.

Use Global Exception Handling.

Use clean architecture.
"""
}

In [30]:
# =====================================================
# PROMPT BUILDER
# =====================================================
#
# Purpose:
# Build the final prompt sent to the LLM.
#
# Flow:
#
# User Request
#      ↓
# Technology Detection
#      ↓
# Template Selection
#      ↓
# Prompt Construction
#
# =====================================================

def build_prompt(
    user_prompt,
    technology
):

    template = PROMPT_TEMPLATES.get(

        technology,

        """
You are a Senior Software Engineer.

Generate production-ready code.

Requirement:

{requirement}
"""
    )

    return template.format(
        requirement=user_prompt
    )

In [31]:
# =====================================================
# LLM WRAPPER
# =====================================================
#
# Purpose:
# Send prompt to LLM.
#
# Future Enhancements:
#
# - Retry logic
# - Token usage tracking
# - Logging
# - Cost tracking
# - Multiple model support
#
# =====================================================

def generate_response(
    prompt
):

    response = client.responses.create(

        model="gpt-4o-mini",

        input=prompt
    )

    return response.output_text

In [32]:
# =====================================================
# CODE GENERATOR AGENT
# =====================================================
#
# Responsibilities:
#
# 1. Understand request
# 2. Detect technology
# 3. Build prompt
# 4. Call LLM
# 5. Return generated code
#
# Agent Workflow:
#
# User Prompt
#      ↓
# Technology Detector
#      ↓
# Prompt Builder
#      ↓
# LLM
#      ↓
# Generated Code
#
# =====================================================

def generate_code(
    user_prompt
):

    # -------------------------------------
    # Detect technology stack
    # -------------------------------------

    technology = detect_technology(
        user_prompt
    )

    # -------------------------------------
    # Build final LLM prompt
    # -------------------------------------

    final_prompt = build_prompt(

        user_prompt=user_prompt,

        technology=technology
    )

    # -------------------------------------
    # Measure execution time
    # -------------------------------------

    start_time = time.time()

    # -------------------------------------
    # Generate code
    # -------------------------------------

    generated_code = generate_response(
        final_prompt
    )

    execution_time = round(

        time.time()
        -
        start_time,

        2
    )

    # -------------------------------------
    # Return structured output
    # -------------------------------------

    return {

        "technology":
        technology,

        "execution_time":
        execution_time,

        "generated_code":
        generated_code
    }

In [33]:
# =====================================================
# AGENT TESTING
# =====================================================
#
# Sample Requests:
#
# Generate Spring Boot CRUD API
#
# Generate FastAPI User Service
#
# Generate React Login Component
#
# Generate Dockerfile
#
# Generate Unit Tests
#
# =====================================================

result = generate_code(

    "Generate Spring Boot CRUD API for Employee"
)

print(

    json.dumps(

        result,

        indent=4
    )
)

{
    "technology": "spring",
    "execution_time": 41.57,
    "generated_code": "Here's a complete production-ready Spring Boot application for a CRUD API for Employee following your requirements:\n\n### 1. `pom.xml`\n\n```xml\n<project xmlns=\"http://maven.apache.org/POM/4.0.0\"\n         xmlns:xsi=\"http://www.w3.org/2001/XMLSchema-instance\"\n         xsi:schemaLocation=\"http://maven.apache.org/POM/4.0.0 http://maven.apache.org/xsd/maven-4.0.0.xsd\">\n    <modelVersion>4.0.0</modelVersion>\n    <groupId>com.example</groupId>\n    <artifactId>employee-api</artifactId>\n    <version>0.0.1-SNAPSHOT</version>\n    <packaging>jar</packaging>\n\n    <name>employee-api</name>\n    <description>Employee CRUD API</description>\n\n    <properties>\n        <java.version>21</java.version>\n        <spring-boot.version>3.0.0</spring-boot.version>\n    </properties>\n\n    <dependencies>\n        <dependency>\n            <groupId>org.springframework.boot</groupId>\n            <artifactId>spr